# Prototype for the openEO pipeline to generate EO4Diversity conform high-resolution temperate forests optimized LAI products based on Sentinel-2 following the OBSGESSION W2.3 benchmarking
<br>
original: EO4Diversity EVI-LAI algorithm to produce the Leaf Area Index product
<br>
The launch of Sentinel-2 provided unprecedented opportunities for high-resolution LAI mapping due to its spectral configuration, which includes multiple red-edge and shortwave infrared (SWIR) bands. Consequently, the Sentinel-2 Simplified Level-2 Prototype (SL2P) algorithm and its implementation in the ESA-SNAP Biophysical Processor (v2.0.2) became a widely used standard for LAI retrieval (Weiss et al., 2021). In parallel, the ESA-funded EO4Diversity project developed and validated an empirical Sentinel-2–based LAI algorithm that links in-situ LAI measurements to the EVI by adapting the approach developed by Boegh et al. (2002), demonstrating robust performance over the considered temperate forest sites. Building on this EO4Diversity framework, the LAI RSIS-bd algorithm adopted in this ATBD uses the EVI-based empirical approach for LAI retrieval using Sentinel-2 time series, ensuring a computationally efficient, validated solution that is currently calibrated and applied to temperate forest ecosystems, but in principle transferable to other vegetated ecosystems with appropriate re-calibration.
<br><br>
This script produces correct LAI datasets and is the starting point for the openEO UDP development.

### Python package area

In [1]:
from eo_processing.utils.helper import init_connection
from eo_processing.openeo.preprocessing import extract_S2_datacube
from eo_processing.config.settings import get_job_options, get_collection_options, get_advanced_options
from eo_processing.utils.metadata import get_base_metadata
from openeo.processes import array_create

### Parameters to set by user
These parameters will be also the openEO UDP parameters in the final product.

In [2]:
# temporal
start_date = '2025-06-01'
end_date = '2025-07-01'
# temporal binning and aggregator
binning_period = 'month'
temp_aggregator = 'mean'
# spatial (has to be a openEO BBOX dict)
AOI = {'east': 4880000, 'south': 2896000, 'west': 4876000, 'north': 2900000, 'crs': 3035}
# resolution and EPSG of output
out_res = 20.0
out_epsg = 3035

### Step 1: generating of cloud-free Sentinel-2 observations for the temporal and spatial domain
- possible improvements:
  - load temperate bio-geographic forest zone
  - load a forest mask
  - mask the Sentinel-2 observations with the temperate forest mask

In [3]:
# establish connection to openEO backend
provider = 'cdse'
connection = init_connection(provider)

Authenticated using refresh token.


In [17]:
# use the eo_processing package to prepare the Sentinel-2 time series for the LAI calculation
processing_options = get_advanced_options(
    provider=provider,
    target_crs=out_epsg,
    resolution=out_res,
    ts_interpolation=False,
    ts_interval=None,
    slc_masking='mask_scl_dilation',
    S2_max_cloud_cover=95,
    S2_bands=["B02", "B04", "B08"],
    skip_check_S2=True,
)
collection_options = get_collection_options(provider=provider)
job_options = get_job_options(provider=provider, task='raw_extraction')

s2_cube = extract_S2_datacube(connection,
                              AOI,
                              start_date,
                              end_date,
                              **collection_options,
                              **processing_options)

### Step 2: generation of the LAI
- EVI is calculated from Sentinel-2 reflectance data using the following expression (Huete et al., 2002): <br>
EVI =2.5 ×(B8−B4)/((B8+6.0×B4−7.5×B2)+1.0) <br>
  , where B8 is the NIR spectral band, B4 is the red band and B2 is the blue band.
- The empirical relationship between LAI and EVI is then expressed as a linear regression model calibrated from field measurements:  <br>
LAI = 4.0543 * EVI + 1.7901
- clamp the LAI values to the range of 0.0 - 7.0 m² m⁻² (values outside this range should be masked out)
- scale the data into Uint8 data type
- The resulting LAI values represent the one-sided green leaf area per unit ground area (m² m⁻²).

In [18]:
# we use the base functionality of openEO
def compute_LAI(bands):
    # select bands
    B2 = bands["B02"] * 0.0001
    B4 = bands["B04"] * 0.0001
    B8 = bands["B08"] * 0.0001

    # create EVI
    EVI = 2.5 * (B8 - B4) / (B8 + 6.0 * B4 - 7.5 * B2 + 1.0)

    # create LAI
    LAI = 4.0543 * EVI + 1.7901
    return array_create([LAI])

LAI_cube = s2_cube.apply_dimension(
    dimension="bands",
    process=compute_LAI,
    context={"parallel": True,
             "TileSize": 128}
).rename_labels("bands", ["LAI"])

# mask out values above 7.5
lai_mask = (LAI_cube < 0) | (LAI_cube > 7.5)
LAI_cube = LAI_cube.mask(lai_mask)

### Step 3: temporal aggregation to user wish
- temporal binning to the user defined period (daily, 10-daily, monthly, annual)
- user of user defined temporal aggregation function (e.g. median, average, maximum)

In [19]:
LAI_cube = LAI_cube.aggregate_temporal_period(period=binning_period, reducer=temp_aggregator)

### Step 4: additional masking

In [20]:
# load the WorldCover 2021 for masking to tree cover
tree_cube = connection.load_collection("ESA_WORLDCOVER_10M_2021_V2",
                                       spatial_extent=AOI,
                                       bands=["MAP"]
                                       )
tree_cube = tree_cube.resample_spatial(projection=out_epsg,
                             resolution=out_res)
tree_cube = tree_cube.drop_dimension('t')
tree_mask = ~ (tree_cube == 10)
LAI_cube = LAI_cube.mask(tree_mask)

# load vector file for temperate forests and also mask
mask_url = 'https://s3.waw4-1.cloudferro.com/swift/v1/obsgession-waw4-1-b2rm8flkntfkatia3zzm7av6pzt3dsmrd2uc87dbvhnml/udp_data/EU_temperate_forests_distribution.parquet'
LAI_cube = LAI_cube.mask_polygon(mask_url)

### Step 5: product creation and saving
- includes metadata ingestion (producer, time, scaling factors)
- COG file format

In [21]:
# force Uint8 and scaling (scal_factor = 1./32)
LAI_cube = LAI_cube.linear_scale_range(0, 7.5, 0, 240)

# prepare metadata
file_meta  = get_base_metadata(project='OBSGESSION', connection=connection)
file_meta.update(description=f'Generation of EO4Diversity conform high-resolution temperate forests optimized LAI products based on Sentinel-2 following the OBSGESSION W2.3 benchmarking.',
                 tiling_grid='LAEA',
                 time_start=start_date,
                 time_end=end_date)
bands_meta = {"LAI": {"description": "LAI",
                              "unit": "m2*m-2",
                              "valid_range": '[0, 240]',
                              "scale": 1./32.,
                              "offset": 0,
                              "nodata_value": 255},
                   }
job = LAI_cube.execute_batch(
    title="LAI test",
    out_format="GTiff",
    job_options=job_options,
    file_metadata=file_meta,
    bands_metadata=bands_meta,
    filename_prefix=f"EO4Diversity_LAI_{binning_period}_{temp_aggregator}_",
)

0:00:00 Job 'j-2603090924514e1d9f00f3e4ce5992c3': send 'start'
0:00:16 Job 'j-2603090924514e1d9f00f3e4ce5992c3': created (progress 0%)
0:00:21 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:00:28 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:00:36 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:00:46 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:00:58 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:01:14 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:01:33 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:01:57 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:02:27 Job 'j-2603090924514e1d9f00f3e4ce5992c3': running (progress N/A)
0:03:04 Job 'j-2603090924514e1d9f00f3e4ce5992c3': finished (progress 100%)


## DOWNLOAD

In [22]:
# get the file
results = job.get_results()
results.download_files(r'C:\Users\buchhorm\Downloads')

[WindowsPath('C:/Users/buchhorm/Downloads/EO4Diversity_LAI_month_mean_2025-06-01-2025-07-01_2025-06-01Z.tif'),
 WindowsPath('C:/Users/buchhorm/Downloads/job-results.json')]

Example jobid for UDP generation: j-2603090924514e1d9f00f3e4ce5992c3

## LIMITATIONS of implementation

- current input can be only Sentinel-2 L2A data from CDSE - usage of other data sources like existing Sentinel-2 composites is not supported yet
- the current output format is GTiff, but maybe netCDF would be better as soon we have several time stamps in the output
- The current applied "forest mask" is based on WorldCover2021 dataset.... maybe we should switch to Land Cover dataset with annual coverage (e.g. CCI)